<a href="https://colab.research.google.com/github/suhailnoor1821/CRM_CW1_Stock_Segmentation/blob/main/CRM_CW1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1.1 Ticker selection and data acquisition
* Tickers: S&P 500 list from Wikipedia (`pd.read_html`), as in Tutorials 04/05.
* Prices: **yfinance** (used in the seminars). `auto_adjust=False` gives both `Adj Close` (for returns and beta) and `Close` (for Monetary); `actions=True` adds dividends and splits for the corporate-action checks.
* Benchmark: `^GSPC`. Period: 1 Jan 2021 to 1 Jan 2025 (end date exclusive).
* **Snapshot:** prices change over time, so the data were downloaded **once on 14 Nov 2026** and saved in the GitHub repository (`data/data_snapshot_2026-11-14`). Every run loads these saved files; the download code only runs if the files are missing.

In [27]:
# Install the yfinance library (used to download share prices)
!pip install -q yfinance

In [28]:
# Import the libraries we need
import os
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Study dates: 1 Jan 2021 to 1 Jan 2025 (the end date itself is not included)
start = '2021-01-01'
end = '2025-01-01'
retrieval_date = '2026-11-14'      # the day we downloaded the data

# Get the saved data (snapshot) from the GitHub repository, so anyone can run this notebook
folder = '/content/CRM_CW1_Stock_Segmentation'
if not os.path.exists(folder):
    !git clone -q https://github.com/suhailnoor1821/CRM_CW1_Stock_Segmentation.git {folder}
snapshot_dir = f'{folder}/data/data_snapshot_{retrieval_date}'
output_dir = f'{folder}/outputs'
os.makedirs(snapshot_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

# Show the package versions (the brief asks for them)
print('pandas', pd.__version__, '| numpy', np.__version__, '| yfinance', yf.__version__)

pandas 2.2.3 | numpy 2.4.6 | yfinance 0.2.66


In [30]:
# Task 1.1 Part 1: Get the list of S&P 500 companies from Wikipedia
# This runs only the first time. After that, the saved list is used
ticker_file = f'{snapshot_dir}/sp500_constituents.csv'

if not os.path.exists(ticker_file):
    # Wikipedia blocks requests without a User-Agent, so we send one
    url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
    table = pd.read_html(url, storage_options={"User-Agent": "Mozilla/5.0"})
    sp500_df = table[0]
    # Yahoo writes BRK.B as BRK-B, so change '.' to '-'
    sp500_df['Ticker'] = sp500_df['Symbol'].str.replace('.', '-', regex=False)
    sp500_df['Retrieval_Date'] = retrieval_date
    sp500_df.to_csv(ticker_file, index=False)

# Load the saved list
sp500_df = pd.read_csv(ticker_file, keep_default_na=False)
tickers = sp500_df['Ticker'].tolist()

print('Total tickers extracted:', len(tickers), '| retrieved on', retrieval_date)
sp500_df[['Symbol', 'Ticker', 'Security', 'GICS Sector', 'Date added']].head(10)

Total tickers extracted: 503 | retrieved on 2026-11-14


,Symbol,Ticker,Security,GICS Sector,Date added
0,MMM,MMM,3M,Industrials,1957-03-04
1,AOS,AOS,A. O. Smith,Industrials,2017-07-26
2,ABT,ABT,Abbott Laboratories,Health Care,1957-03-04
3,ABBV,ABBV,AbbVie,Health Care,2012-12-31
4,ACN,ACN,Accenture,Information Technology,2011-07-06
5,ADBE,ADBE,Adobe Inc.,Information Technology,1997-05-05
6,AMD,AMD,Advanced Micro Devices,Information Technology,2017-03-20
7,AES,AES,AES Corporation,Utilities,1998-10-02
8,AFL,AFL,Aflac,Financials,1999-05-28
9,A,A,Agilent Technologies,Health Care,2000-06-05


In [31]:
# Task 1.1 Part 2: Daily prices and volume for every share and for the S&P 500 (^GSPC)
# The download runs only if the saved files are missing. After that, the saved files are used
if not os.path.exists(f'{snapshot_dir}/adj_close_daily.csv.gz'):
    df = yf.download(tickers, start=start, end=end, auto_adjust=False, actions=True)
    df['Adj Close'].to_csv(f'{snapshot_dir}/adj_close_daily.csv.gz')
    df['Close'].to_csv(f'{snapshot_dir}/close_daily.csv.gz')
    df['Volume'].to_csv(f'{snapshot_dir}/volume_daily.csv.gz')
    df['Dividends'].to_csv(f'{snapshot_dir}/dividends_daily.csv.gz')
    df['Stock Splits'].to_csv(f'{snapshot_dir}/splits_daily.csv.gz')

    # Download the S&P 500 index (the benchmark)
    market = yf.download('^GSPC', start=start, end=end, auto_adjust=False)
    market.columns = market.columns.get_level_values(0)
    market[['Adj Close', 'Close', 'Volume']].to_csv(f'{snapshot_dir}/gspc_daily.csv.gz')

# Small helper that reads one saved file
def load(name):
    return pd.read_csv(f'{snapshot_dir}/{name}_daily.csv.gz', index_col=0, parse_dates=True)

adj_close = load('adj_close')
close = load('close')
volume = load('volume')
dividends = load('dividends')
splits = load('splits')
gspc = load('gspc')        # S&P 500 index (benchmark)

print('Shares:', adj_close.shape[1], '| trading days:', adj_close.shape[0],
      '| first:', adj_close.index.min().date(), '| last:', adj_close.index.max().date())

Shares: 503 | trading days: 1005 | first: 2021-01-04 | last: 2024-12-31


In [32]:
# Find tickers where Yahoo Finance gave no data at all
no_data = [t for t in tickers if t not in adj_close.columns or adj_close[t].isna().all()]
print('Tickers with no data:', len(no_data), no_data)

Tickers with no data: 5 ['FDXF', 'HONA', 'Q', 'SNDK', 'VYLR']


## 1.2 Data validation and alignment
The tutorial code filled gaps with `ffill`/`bfill`. The brief says **do not silently fill missing prices**, so each share is checked instead and **excluded** (with a reason) if it has:
* no data, or a history that starts after 4 Jan 2021 / ends before 31 Dec 2024 (unequal history)
* any missing day, a zero/negative price, or a zero/negative volume
* a price jump on a stock-split day (meaning the prices are not split-adjusted)

In [ ]:
# Task 1.2 Part 1: Check for duplicate dates and missing S&P 500 prices
print('Trading days:', len(gspc), '| first:', gspc.index.min().date(), '| last:', gspc.index.max().date())
print('Duplicate dates - shares:', adj_close.index.duplicated().sum(), '| benchmark:', gspc.index.duplicated().sum())
print('Missing benchmark prices:', gspc['Adj Close'].isna().sum())
print('Benchmark Adj Close equals Close:', np.allclose(gspc['Adj Close'], gspc['Close']))

# Put every share on the same trading days as the S&P 500
# Missing days stay empty - we do NOT fill them
adj_close = adj_close.reindex(gspc.index)
close = close.reindex(gspc.index)
volume = volume.reindex(gspc.index)
dividends = dividends.reindex(gspc.index).fillna(0)   # 0 = no dividend that day
splits = splits.reindex(gspc.index).fillna(0)         # 0 = no split that day

Trading days: 1005 | first: 2021-01-04 | last: 2024-12-31
Duplicate dates - shares: 0 | benchmark: 0
Missing benchmark prices: 0
Benchmark Adj Close equals Close: True


In [ ]:
# Task 1.2 Part 2: Check every share and write down why a share is removed
first_day = gspc.index[0]
last_day = gspc.index[-1]

# One row per share: first date, last date and how many days have problems
checks = pd.DataFrame(index=adj_close.columns)
checks['First_Date'] = adj_close.apply(lambda s: s.first_valid_index())
checks['Last_Date'] = adj_close.apply(lambda s: s.last_valid_index())
checks['Missing_Days'] = (adj_close.isna() | close.isna() | volume.isna()).sum()
checks['Bad_Price_Days'] = ((adj_close <= 0) | (close <= 0)).sum()
checks['Zero_Volume_Days'] = (volume <= 0).sum()
checks['Splits'] = (splits > 0).sum()
checks['Dividends'] = (dividends > 0).sum()

# Stock splits: Yahoo already adjusts prices for splits,
# so the Close price should NOT jump on a split day
daily_change = (close / close.shift(1) - 1).abs()
checks['Change_On_Split_Day'] = daily_change.where(splits > 0).max()

checks = checks.reindex(tickers)      # one row for every ticker on the Wikipedia list

# Give the reason a share is removed (empty text = the share is kept)
def exclusion_reason(row):
    if pd.isna(row['First_Date']):
        return 'No data from Yahoo Finance'
    if row['First_Date'] > first_day:
        return f"Listed after study start ({row['First_Date'].date()})"
    if row['Last_Date'] < last_day:
        return f"History ends early ({row['Last_Date'].date()})"
    if row['Missing_Days'] > 0:
        return f"{int(row['Missing_Days'])} missing days"
    if row['Bad_Price_Days'] > 0:
        return 'Zero or negative price'
    if row['Zero_Volume_Days'] > 0:
        return f"{int(row['Zero_Volume_Days'])} zero-volume days"
    if row['Change_On_Split_Day'] > 0.25:
        return 'Price jump on split day'
    return ''

checks['Exclusion_Reason'] = checks.apply(exclusion_reason, axis=1)
checks = checks.join(sp500_df.set_index('Ticker')[['Security', 'GICS Sector']])
# List of shares that passed every check
included = checks.index[checks['Exclusion_Reason'] == ''].tolist()

print('Tickers on the list:', len(checks), '| retained:', len(included), '| excluded:', len(checks) - len(included))
checks.loc[checks['Exclusion_Reason'] != '', ['Security', 'GICS Sector', 'First_Date', 'Exclusion_Reason']]

Tickers on the list: 503 | retained: 486 | excluded: 17


,Security,GICS Sector,First_Date,Exclusion_Reason
APP,AppLovin,Communication Services,2021-04-15,Listed after study start (2021-04-15)
COIN,Coinbase,Financials,2021-04-14,Listed after study start (2021-04-14)
CEG,Constellation Energy,Utilities,2022-01-19,Listed after study start (2022-01-19)
EXE,Expand Energy,Energy,2021-02-10,Listed after study start (2021-02-10)
FDXF,FedEx Freight,Industrials,NaT,No data from Yahoo Finance
FERG,Ferguson Enterprises,Industrials,2021-01-04,27 zero-volume days
GEHC,GE HealthCare,Health Care,2022-12-15,Listed after study start (2022-12-15)
GEV,GE Vernova,Industrials,2024-03-27,Listed after study start (2024-03-27)
HONA,Honeywell Aerospace,Industrials,NaT,No data from Yahoo Finance
Q,Qnity Electronics,Information Technology,NaT,No data from Yahoo Finance


In [ ]:
# Task 1.2 Part 3: Summary of the checks for the shares we kept
kept = checks.loc[included]
print('Missing days:', int(kept['Missing_Days'].sum()),
      '| bad prices:', int(kept['Bad_Price_Days'].sum()),
      '| zero volumes:', int(kept['Zero_Volume_Days'].sum()))
print('Shares with a stock split:', int((kept['Splits'] > 0).sum()),
      '| largest Close change on a split day:', round(kept['Change_On_Split_Day'].max(), 4))
print('Shares paying dividends:', int((kept['Dividends'] > 0).sum()))

# Adj Close / Close should only change on a dividend or split day
# (0 below means every change is explained)
factor_change = (adj_close / close).pct_change(fill_method=None).abs() > 0.001
unexplained = factor_change & (dividends == 0) & (splits == 0)
print('Adjustment changes NOT on a dividend/split day:', int(unexplained[included].sum().sum()))

# Save the check results
checks.to_csv(f'{output_dir}/validation_log.csv')

# Show all stock splits for the shares we kept
split_list = splits[included].stack()
split_list[split_list > 0].rename('Split_Ratio').to_frame()

Missing days: 0 | bad prices: 0 | zero volumes: 0
Shares with a stock split: 53 | largest Close change on a split day: 0.0766
Shares paying dividends: 392
Adjustment changes NOT on a dividend/split day: 0


,,Split_Ratio
Date,,
2021-03-05,APH,2.000
2021-04-01,SHW,3.000
2021-06-03,MRK,1.048
2021-06-28,CSGP,10.000
2021-06-29,CSX,3.000
...,...,...
2024-10-03,LRCX,10.000
2024-12-04,ANET,4.000
2024-12-13,ETR,2.000


**Survivorship bias.** The ticker list is today's S&P 500, applied back to 2021–2024. Companies that left the index during the period (acquired, delisted, removed) are missing, and companies that grew into the index are included, so the sample favours past winners. Shares listed after January 2021 are also excluded, so the retained set is "current members with a full 2021–2024 history", not the historical index.